# EKF Velocity Fusion — Demonstration Notebook

**Sprint 023, Ticket T006**

This notebook demonstrates the 5-state EKF (x, y, theta, v, omega) by replaying
a synthetic TLM log through three configurations:

- **Encoder-only**: predict from wheel encoder deltas, no OTOS updates
- **+OTOS position**: also fuse OTOS x/y via `update_position()`
- **+OTOS velocity**: also fuse OTOS/encoder velocity via `update_velocity()`

**Required data file**: `tests/dev/fixtures/tlm_log_sample.txt`
(committed to the repository; no hardware required)

**Usage**: Run all cells top-to-bottom (`Cell → Run All`).
No hardware connection needed. All data comes from the committed fixture log.

**Source**: `tests/dev/ekf_replay.py` (replay harness) and
`tests/dev/test_ekf.py` (Python EKF mirror of `source/control/EKF.cpp`).

In [ ]:
"""Setup: add project root to path and import the replay harness."""
import sys
import os
import importlib.util
import math

# Resolve project root (two directories above host_tests/)
_NB_DIR = os.path.dirname(os.path.abspath('__file__'))
_PROJECT_ROOT = os.path.dirname(_NB_DIR)
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

# Load ekf_replay module via direct path (avoids package import issues)
_replay_path = os.path.join(_PROJECT_ROOT, 'tests', 'dev', 'ekf_replay.py')
_spec = importlib.util.spec_from_file_location('ekf_replay', _replay_path)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
replay_tlm_log = _mod.replay_tlm_log

print('ekf_replay loaded from:', _replay_path)

In [ ]:
"""Data: load the committed fixture log and run replay in three configurations."""
LOG_FILE = os.path.join(_PROJECT_ROOT, 'tests', 'dev', 'fixtures', 'tlm_log_sample.txt')
assert os.path.exists(LOG_FILE), f'Fixture log not found: {LOG_FILE}'
print(f'Log file: {LOG_FILE}')

# Count valid TLM lines for reference
with open(LOG_FILE) as f:
    n_lines = sum(1 for ln in f if ln.strip() and not ln.strip().startswith('#'))
print(f'TLM lines: {n_lines}')

# Run replay in three configurations
frames_enc  = replay_tlm_log(LOG_FILE, encoder_only=True,  otos_position=False, otos_velocity=False)
frames_pos  = replay_tlm_log(LOG_FILE, encoder_only=False, otos_position=True,  otos_velocity=False)
frames_vel  = replay_tlm_log(LOG_FILE, encoder_only=False, otos_position=True,  otos_velocity=True)

print(f'Encoder-only frames: {len(frames_enc)}')
print(f'+OTOS position frames: {len(frames_pos)}')
print(f'+OTOS velocity frames: {len(frames_vel)}')

In [ ]:
"""Replay: extract time series for plotting."""

def unpack_frames(frames):
    """Unpack a list of ReplayFrame named tuples into parallel arrays."""
    t_ms   = [f.t_ms   for f in frames]
    xs     = [f.x      for f in frames]
    ys     = [f.y      for f in frames]
    thetas = [f.theta  for f in frames]
    vs     = [f.v      for f in frames]
    omegas = [f.omega  for f in frames]
    p00    = [f.p_diag[0] for f in frames]
    p33    = [f.p_diag[3] for f in frames]
    return t_ms, xs, ys, thetas, vs, omegas, p00, p33

t_enc, x_enc, y_enc, th_enc, v_enc, om_enc, p00_enc, p33_enc = unpack_frames(frames_enc)
t_pos, x_pos, y_pos, th_pos, v_pos, om_pos, p00_pos, p33_pos = unpack_frames(frames_pos)
t_vel, x_vel, y_vel, th_vel, v_vel, om_vel, p00_vel, p33_vel = unpack_frames(frames_vel)

# Load the raw OTOS pose from the fixture for reference.
from host.robot_radio.robot.protocol import parse_tlm
raw_pose_x, raw_pose_y, raw_pose_th = [], [], []
raw_twist_v, raw_twist_om = [], []
with open(LOG_FILE) as f:
    for ln in f:
        if ln.strip() and not ln.strip().startswith('#'):
            fr = parse_tlm(ln)
            if fr and fr.pose:
                raw_pose_x.append(float(fr.pose[0]))
                raw_pose_y.append(float(fr.pose[1]))
                raw_pose_th.append(fr.pose[2] * math.pi / 18000.0)  # cdeg→rad
            if fr and fr.twist:
                raw_twist_v.append(float(fr.twist[0]))
                raw_twist_om.append(float(fr.twist[1]) * 0.001)     # mrad/s→rad/s

print(f'Raw OTOS pose samples: {len(raw_pose_x)}')
print(f'Raw twist samples: {len(raw_twist_v)}')

In [ ]:
"""Plot 1: x-y trajectory overlay for the three configurations."""
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(x_enc, y_enc, 'b-o', markersize=3, linewidth=1.5, label='Encoder only')
ax.plot(x_pos, y_pos, 'g-s', markersize=3, linewidth=1.5, label='+OTOS position')
ax.plot(x_vel, y_vel, 'r-^', markersize=3, linewidth=1.5, label='+OTOS velocity')
ax.plot(raw_pose_x, raw_pose_y, 'k--', linewidth=1, alpha=0.5, label='Raw OTOS (reference)')
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_title('Plot 1: x-y Trajectory Overlay\n(encoder-only vs +OTOS position vs +OTOS velocity)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal', adjustable='datalim')
plt.tight_layout()
plt.show()
print('Plot 1 done.')

In [ ]:
"""Plot 2: position error vs time (distance from raw OTOS reference)."""

def pos_error(xs, ys, ref_x, ref_y):
    """Euclidean distance from reference trajectory at each time step."""
    n = min(len(xs), len(ref_x))
    return [math.sqrt((xs[i]-ref_x[i])**2 + (ys[i]-ref_y[i])**2) for i in range(n)]

err_enc = pos_error(x_enc, y_enc, raw_pose_x, raw_pose_y)
err_pos = pos_error(x_pos, y_pos, raw_pose_x, raw_pose_y)
err_vel = pos_error(x_vel, y_vel, raw_pose_x, raw_pose_y)
t_plot = t_enc[:len(err_enc)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_plot, err_enc[:len(t_plot)], 'b-', linewidth=1.5, label='Encoder only')
ax.plot(t_plot, err_pos[:len(t_plot)], 'g-', linewidth=1.5, label='+OTOS position')
ax.plot(t_plot, err_vel[:len(t_plot)], 'r-', linewidth=1.5, label='+OTOS velocity')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Position error vs OTOS reference (mm)')
ax.set_title('Plot 2: Position Error vs Time')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Plot 2 done.')

In [ ]:
"""Plot 3: fused v and omega vs raw encoder-rate and OTOS twist."""

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

# v (linear velocity)
ax1.plot(t_enc, v_enc, 'b-', linewidth=1.5, alpha=0.8, label='EKF v (enc only)')
ax1.plot(t_vel, v_vel, 'r-', linewidth=1.5, label='EKF v (+OTOS vel)')
if raw_twist_v:
    t_tw = t_enc[:len(raw_twist_v)]
    ax1.plot(t_tw, raw_twist_v, 'k--', linewidth=1, alpha=0.5, label='Raw twist v (mm/s)')
ax1.set_ylabel('v (mm/s)')
ax1.set_title('Plot 3: Fused v and omega vs Raw Measurements')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# omega (angular velocity)
ax2.plot(t_enc, om_enc, 'b-', linewidth=1.5, alpha=0.8, label='EKF omega (enc only)')
ax2.plot(t_vel, om_vel, 'r-', linewidth=1.5, label='EKF omega (+OTOS vel)')
if raw_twist_om:
    t_tw = t_enc[:len(raw_twist_om)]
    ax2.plot(t_tw, raw_twist_om, 'k--', linewidth=1, alpha=0.5, label='Raw twist omega (rad/s)')
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('omega (rad/s)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Plot 3 done.')

In [ ]:
"""Plot 4: P[0][0] (position uncertainty) and P[3][3] (velocity uncertainty) vs time.

Expected behaviour:
  - P[0][0] grows during predict-only stretches and shrinks at OTOS position updates.
  - P[3][3] grows during predict-only stretches and shrinks at velocity updates.
  - Encoder-only config: P[0][0] grows without bound (no correction).
"""
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

ax1.plot(t_enc, p00_enc, 'b-', linewidth=1.5, label='Encoder only')
ax1.plot(t_pos, p00_pos, 'g-', linewidth=1.5, label='+OTOS position')
ax1.plot(t_vel, p00_vel, 'r-', linewidth=1.5, label='+OTOS velocity')
ax1.set_ylabel('P[0][0] — x uncertainty (mm²)')
ax1.set_title('Plot 4: Covariance Diagonal — Position and Velocity Uncertainty vs Time')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

ax2.plot(t_enc, p33_enc, 'b-', linewidth=1.5, label='Encoder only')
ax2.plot(t_vel, p33_vel, 'r-', linewidth=1.5, label='+OTOS velocity')
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('P[3][3] — v uncertainty (mm/s)²')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Plot 4 done.')

In [ ]:
"""Plot 5: theta error vs time — heading drift between camera fixes.

Shows how theta deviates from the raw OTOS heading reference over time.
Camera fixes (OTOS position updates with large jumps) indirectly correct
theta via cross-covariance, but the correction is limited when the filter
is confident about heading (P[2][2] small).
"""

def heading_error(est_thetas, ref_thetas):
    """Heading error (wrapped to (-pi, pi]), in degrees."""
    n = min(len(est_thetas), len(ref_thetas))
    import math as _m
    def wpi(a): return _m.atan2(_m.sin(a), _m.cos(a))
    return [wpi(est_thetas[i] - ref_thetas[i]) * 180.0 / math.pi for i in range(n)]

h_err_enc = heading_error(th_enc, raw_pose_th)
h_err_pos = heading_error(th_pos, raw_pose_th)
h_err_vel = heading_error(th_vel, raw_pose_th)
t_h = t_enc[:len(h_err_enc)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_h, h_err_enc, 'b-', linewidth=1.5, label='Encoder only')
ax.plot(t_h, h_err_pos[:len(t_h)], 'g-', linewidth=1.5, label='+OTOS position')
ax.plot(t_h, h_err_vel[:len(t_h)], 'r-', linewidth=1.5, label='+OTOS velocity')
ax.axhline(0, color='k', linewidth=0.5)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Theta error (degrees)')
ax.set_title('Plot 5: Heading Error vs Time')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Plot 5 done.')

## Note: Heading-Axis Weakness

**theta is corrected only indirectly via position cross-covariance and by camera
resets; its absolute drift is fully removed only by a camera fix (SI command).
OTOS heading fusion remains future work.**

In this EKF design the observation model for `update_position()` has
`H = [[1,0,0,0,0],[0,1,0,0,0]]` — it directly observes x and y but NOT theta.
The correction to theta at an OTOS position update flows only through
`P[2][0]` and `P[2][1]` (the cross-covariance terms between theta and x/y).
These terms are typically small because theta evolves mostly orthogonally to
position in the arc-segment model.

As a result:
- Between camera fixes, heading drifts at the encoder integration rate.
- A position fix corrects theta only weakly (proportional to `P[2][x]`).
- A full `set_pose()` (SI command) resets theta directly and is the only
  mechanism that removes accumulated heading error entirely.

Future improvement: add a direct theta observation channel
(`update_heading(theta_otos)`) with `H = [0,0,1,0,0]`, which would allow
the OTOS heading reading to correct theta directly.
This is left as future work (sprint 024+) because the OTOS heading
is less reliable than its position reading and requires its own
calibration and gating strategy.